In [3]:
import re
import unicodedata

with open("../../data/raw/taleOf2Cities.txt", "r", encoding="utf-8") as f:
   text = f.read()

text = unicodedata.normalize("NFKC", text)

text = re.sub(r"\[Illustration:.*?\]\]?", "", text, flags=re.DOTALL)
text = re.sub(r"_([^_]+)_", r"\1", text)
text = re.sub(r"\n\s*CHAPTER\s+[IVXLC0-9]+\.*\s*\n", "\n", text, flags=re.IGNORECASE)

words = text.split()
chunks = []
for i in range (0, len(words), 160):
    chunk = words[i:i+160]
    if len(chunk) >= 120:
        chunks.append(" ".join(chunk))

with open("../../data/processed/taleOf2Cities.txt", "w", encoding="utf-8") as f:
    f.write(text)

with open("../../data/processed/taleOf2Citieschunked.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(chunks))

In [ ]:
import google.generativeai as genai
import time
import os
from getpass import getpass

# Configure Gemini API
api_key = os.getenv("GEMINI_API_KEY")

# If not in environment, prompt the user to enter it
if not api_key:
    print("GEMINI_API_KEY not found in environment.")
    print("Paste your API key below (it will be hidden):")
    api_key = getpass("API Key: ")

if not api_key:
    raise ValueError("API key is required to continue.")

genai.configure(api_key=api_key)

# Read topics
with open("../../data/processed/2citiestopics.txt", "r") as f:
    twoCitiestopics = [line.strip() for line in f if line.strip()]


print(f"The Tale of 2 Cities topics: {len(twoCitiestopics)}")
print("API configured successfully!")

In [ ]:
with open("../../data/processed/taleOf2Citieschunked.txt", "r", encoding="utf-8") as f:
    cityexamples = [p.strip() for p in f.read().split("\n\n") if p.strip()][:50]

import random
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Thread-safe list helper
lock = threading.Lock()

# Class 2: Plain Gemini generation on the same topics (no style mimicry)
def generate_class2_paragraphs(topics, output_file, num_paragraphs=500):
    paragraphs = []
    model = genai.GenerativeModel("gemini-2.5-flash-lite")
    topics_str = ", ".join(topics)

    def _generate_one_class2(idx):
        try:
            prompt = f"""Write a single paragraph (100-200 words) about these topics: {topics_str}
Rules:
- Just write the paragraph, no introductions or meta-commentary
- Vary your sentence lengths naturally
- Don't be very complex, use relatively simple non nested sentences, with very little nested sentences occasionally
"""
            response = model.generate_content(prompt)
            text = response.text.strip()
            if text.startswith('"') and text.endswith('"'):
                text = text[1:-1]
            return (idx, text)
        except Exception as e:
            print(f"Error generating paragraph {idx}: {e}")
            time.sleep(5)
            return None

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(_generate_one_class2, i): i for i in range(num_paragraphs)}
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                with lock:
                    paragraphs.append(result)
                    count = len(paragraphs)
                print(f"Generated {count}/{num_paragraphs} paragraphs")
                if count % 50 == 0:
                    with lock:
                        sorted_p = [t for _, t in sorted(paragraphs)]
                    os.makedirs("../../data/generated", exist_ok=True)
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write("\n\n".join(sorted_p))
                    print(f"✓ Saved {count} paragraphs to {output_file}")

    # Final save (sorted by original index)
    sorted_paragraphs = [t for _, t in sorted(paragraphs)]
    os.makedirs("../../data/generated", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sorted_paragraphs))
    print(f"✓ Final save: {len(sorted_paragraphs)} paragraphs to {output_file}")


# Class 3: Gemini generation mimicking Charles Dickens's style
def generate_class3_paragraphs(topics, output_file, num_paragraphs=500):
    paragraphs = []
    model = genai.GenerativeModel("gemini-2.5-flash-lite")
    topics_str = ", ".join(topics)

    def _generate_one_class3(idx):
        try:
            selected_examples = random.sample(cityexamples, min(3, len(cityexamples)))
            examples_text = "\n\n---\n\n".join(selected_examples)

            prompt = f"""You are an expert literary mimic specializing in Charles Dickens's style. Study these authentic excerpts from A Tale of Two Cities:

=== AUTHENTIC EXAMPLES ===
{examples_text}
=== END EXAMPLES ===

Now write a NEW paragraph (100-200 words) about: {topics_str}

Write a very readable and simple paragraph.
DO not over describe.
Don't use very complex or nested sentences, keep it relatively simple.
Try to have a high hapax legomena.
CHARLES DICKENS STYLE REQUIREMENTS:
1. Use his characteristic vivid imagery and social commentary
2. Use decent amount of double quotes and dialogues
3. Include his trademark sharp irony and dark humor
4. Use semicolons to connect related thoughts gracefully along with some (very few but some) em-dashes in his style
8. Keep readability at Flesch-Kincaid grade level ~9
9. Avoid excessive nesting - favor clarity over complexity

Output ONLY the paragraph. No explanations, no quotation marks, no meta-commentary."""

            response = model.generate_content(prompt)
            text = response.text.strip()
            if text.startswith('"') and text.endswith('"'):
                text = text[1:-1]
            return (idx, text)
        except Exception as e:
            print(f"Error generating paragraph {idx}: {e}")
            time.sleep(5)
            return None

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(_generate_one_class3, i): i for i in range(num_paragraphs)}
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                with lock:
                    paragraphs.append(result)
                    count = len(paragraphs)
                print(f"Generated {count}/{num_paragraphs} paragraphs")
                if count % 50 == 0:
                    with lock:
                        sorted_p = [t for _, t in sorted(paragraphs)]
                    os.makedirs("../../data/generated", exist_ok=True)
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write("\n\n".join(sorted_p))
                    print(f"✓ Saved {count} paragraphs to {output_file}")

    # Final save (sorted by original index)
    sorted_paragraphs = [t for _, t in sorted(paragraphs)]
    os.makedirs("../../data/generated", exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n\n".join(sorted_paragraphs))
    print(f"✓ Final save: {len(sorted_paragraphs)} paragraphs to {output_file}")

In [ ]:
generate_class2_paragraphs(twoCitiestopics, "../../data/generated/2cities_class2.txt", num_paragraphs=500)